# Easy Agent — Orchestration Layer (Python Notebook)

This notebook walks through the **four subsystems** of Easy Agent's orchestration layer:

| Subsystem | Concern |
|---|---|
| **Agents** | Registering, loading, resolving, and executing sub-agents |
| **Context** | System-prompt assembly, memory, compaction, plan mode |
| **Session** | JSONL transcript persistence and restoration |
| **State** | In-memory reactive stores (async agents, notifications) |

Every cell is self-contained Python — no project imports. Logic mirrors the TypeScript source in `src/agents/`, `src/context/`, `src/session/`, and `src/state/`.

---

## 1. Setup & Path Discovery

In [ ]:
from pathlib import Path
import os, re, json, hashlib, uuid, time, asyncio, threading, textwrap
from dataclasses import dataclass, field
from typing import Optional, Callable, Any
from datetime import datetime, timezone

_cwd = Path(".").resolve()
PROJECT_ROOT = _cwd
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "package.json").exists() or (PROJECT_ROOT / "pyproject.toml").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise FileNotFoundError(f"Could not find package.json or pyproject.toml starting from {_cwd}")

EASY_AGENT_HOME = Path.home() / ".easy-agent"
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"EASY_AGENT_HOME : {EASY_AGENT_HOME}")

## 2. Data Models

Core type definitions from `src/agents/types.ts`.
`getSystemPrompt` is a **callable** (not a string) so built-in and composed prompts can be constructed lazily.

In [ ]:
from typing import Literal

AgentSource = Literal["built-in", "user", "project"]
AgentPermissionMode = Literal["default", "plan", "auto"]
AgentIsolation = Literal["none", "worktree"]

@dataclass
class AgentDefinition:
    agentType: str            # unique id — value passed as subagent_type
    whenToUse: str            # shown in system prompt for model discovery
    source: AgentSource       # where the definition came from
    getSystemPrompt: Callable[[], str] = field(default=lambda: "", repr=False)
    tools: Optional[list[str]] = None          # None / ['*'] → wildcard
    disallowedTools: list[str] = field(default_factory=list)
    model: Optional[str] = None
    maxTurns: Optional[int] = None
    permissionMode: Optional[AgentPermissionMode] = None
    isolation: Optional[AgentIsolation] = None
    filePath: Optional[str] = None             # disk path for custom agents

@dataclass
class AgentRunResult:
    agentType: str
    finalText: str
    messages: list            # full message history
    totalToolUseCount: int
    totalDurationMs: int
    totalTokens: int
    inputTokens: int
    outputTokens: int
    turnCount: int
    reason: str               # LoopTerminationReason
    warnings: list[str] = field(default_factory=list)

print("Data models defined: AgentDefinition, AgentRunResult")

## 3. Agent Registry

`src/agents/registry.ts`

A **write-once, read-many** in-memory dict. Built-ins are loaded first; project-scope agents loaded last → `project > user > built-in` collision resolution (last-writer-wins on `dict.__setitem__`).

In [ ]:
class AgentRegistry:
    """Module-level singleton that holds all loaded AgentDefinitions."""

    def __init__(self):
        self._agents: dict[str, AgentDefinition] = {}
        self._initialized = False

    def set_agents(self, definitions: list[AgentDefinition]) -> None:
        """Bulk-replace the registry. Called ONCE at startup by bootstrapAgents."""
        self._agents.clear()
        for d in definitions:
            self._agents[d.agentType] = d   # last-writer-wins → project overrides built-in
        self._initialized = True

    def is_initialized(self) -> bool:
        return self._initialized

    def find_agent(self, agent_type: str) -> Optional[AgentDefinition]:
        return self._agents.get(agent_type)

    def get_all_agents(self) -> list[AgentDefinition]:
        return list(self._agents.values())

    def clear_agents(self) -> None:
        self._agents.clear()
        self._initialized = False

# Module-level singleton
REGISTRY = AgentRegistry()
print("AgentRegistry singleton created")

## 4. Built-in Agent Definitions

`src/agents/builtIn/explore.ts` and `src/agents/builtIn/generalPurpose.ts`

| Agent | Tool restriction | Strategy |
|---|---|---|
| **Explore** | `disallowedTools: [Write, Edit, MemoryWrite]` | belt-and-suspenders: prompt + structural deny |
| **general-purpose** | wildcard (`tools=None`) | inherits full parent pool |

In [ ]:
EXPLORE_SYSTEM_PROMPT = textwrap.dedent("""
    You are a read-only code-exploration sub-agent for Easy Agent.

    === READ-ONLY MODE — DO NOT MODIFY ANY FILES ===

    Your toolset is limited to: Read, Grep, Glob, and read-only Bash
    (ls, cat, head, tail, git status, git log, git diff, find, etc.).

    When finished, return a concise report covering:
    - Where the relevant code lives (file paths + line ranges).
    - The patterns and conventions it follows.
    - Any gotchas the main agent needs to know before making changes.
""").strip()

GENERAL_PURPOSE_SYSTEM_PROMPT = textwrap.dedent("""
    You are a general-purpose sub-agent for Easy Agent.
    Complete the delegated task fully and correctly using any available tools.
    Reply with a concise report of what you did and any key findings.
""").strip()

EXPLORE_AGENT = AgentDefinition(
    agentType="Explore",
    whenToUse=(
        "Read-only code search and exploration agent. Use when you need to "
        "find files, search code, or trace usages WITHOUT making changes."
    ),
    source="built-in",
    disallowedTools=["Write", "Edit", "MemoryWrite"],
    getSystemPrompt=lambda: EXPLORE_SYSTEM_PROMPT,
)

GENERAL_PURPOSE_AGENT = AgentDefinition(
    agentType="general-purpose",
    whenToUse=(
        "General-purpose sub-agent for delegating focused subtasks. "
        "Inherits the parent's full tool set."
    ),
    source="built-in",
    tools=None,   # wildcard
    getSystemPrompt=lambda: GENERAL_PURPOSE_SYSTEM_PROMPT,
)

def get_built_in_agents() -> list[AgentDefinition]:
    return [EXPLORE_AGENT, GENERAL_PURPOSE_AGENT]

print("Built-ins:", [a.agentType for a in get_built_in_agents()])
print("Explore disallowedTools:", EXPLORE_AGENT.disallowedTools)
print("general-purpose tools (None=wildcard):", GENERAL_PURPOSE_AGENT.tools)

## 5. Agent Loading from Disk

`src/agents/loadAgentsDir.ts`

Custom agents are `.md` files with YAML frontmatter. Required fields: `name`, `description`. The markdown body becomes the system prompt.

**Two scopes scanned (in parallel):**
1. `~/.easy-agent/agents/`  → `"user"` source
2. `<cwd>/.easy-agent/agents/` → `"project"` source

> **Note:** YAML parsing is simplified here (regex-based). The real loader delegates to `splitFrontmatter()` which uses `js-yaml`.

In [ ]:
def _as_string(value) -> Optional[str]:
    if isinstance(value, str):
        t = value.strip()
        return t if t else None
    if isinstance(value, (int, float, bool)):
        return str(value)
    return None

def _as_string_array(value) -> list[str]:
    if isinstance(value, list):
        return [v.strip() for v in value if isinstance(v, str) and v.strip()]
    if isinstance(value, str):
        return [s.strip() for s in value.split(",") if s.strip()]
    return []

def _as_positive_int(value) -> Optional[int]:
    if isinstance(value, int) and value > 0:
        return value
    if isinstance(value, str):
        try:
            n = int(value.strip())
            return n if n > 0 else None
        except ValueError:
            return None
    return None

def _as_permission_mode(value) -> Optional[AgentPermissionMode]:
    return value if value in ("default", "plan", "auto") else None

def _as_isolation(value) -> Optional[AgentIsolation]:
    return value if value in ("worktree", "none") else None

def split_frontmatter(raw: str) -> tuple[dict, str, Optional[str]]:
    """Returns (frontmatter_dict, body, parse_error)."""
    m = re.match(r'^---\n([\s\S]*?)\n---\n?', raw)
    if not m:
        return {}, raw, "no frontmatter block found"
    fm_text = m.group(1)
    body = raw[m.end():]
    fm = {}
    for line in fm_text.splitlines():
        if ":" in line:
            k, _, v = line.partition(":")
            fm[k.strip()] = v.strip().strip('"').strip("'")
    return fm, body, None

def load_agents_from_markdown(raw: str, file_path: str, source: AgentSource) -> tuple[Optional[AgentDefinition], Optional[str]]:
    """Parse one .md file into an AgentDefinition (or return a warning)."""
    fm, body, err = split_frontmatter(raw)
    if err:
        return None, f"Skipping {file_path}: invalid frontmatter ({err})"

    name = _as_string(fm.get("name"))
    description = _as_string(fm.get("description"))
    if not name:
        return None, f"Skipping {file_path}: missing required 'name' field"
    if not description:
        return None, f"Skipping {file_path}: missing required 'description' field"

    system_prompt = body.strip()
    if not system_prompt:
        return None, f"Skipping {file_path}: empty body — agent needs a system prompt"

    tools = _as_string_array(fm.get("tools", []))
    disallowed = _as_string_array(fm.get("disallowedTools", fm.get("disallowed_tools", [])))
    model = _as_string(fm.get("model"))
    max_turns = _as_positive_int(fm.get("maxTurns", fm.get("max_turns")))
    perm_mode = _as_permission_mode(fm.get("permissionMode", fm.get("permission_mode")))
    isolation = _as_isolation(fm.get("isolation"))

    agent = AgentDefinition(
        agentType=name,
        whenToUse=description,
        source=source,
        tools=tools if tools else None,
        disallowedTools=disallowed,
        model=model,
        maxTurns=max_turns,
        permissionMode=perm_mode,
        isolation=isolation,
        filePath=file_path,
        getSystemPrompt=lambda sp=system_prompt: sp,
    )
    return agent, None

# Demo — parse a synthetic custom agent
SAMPLE_AGENT_MD = """---
name: "code-reviewer"
description: "Reviews code for quality issues. Use when user asks for a review."
tools: "Read,Grep,Glob"
disallowedTools: "Write,Edit"
model: "claude-haiku-4-5-20251001"
maxTurns: 12
permissionMode: "default"
---
You are a senior code reviewer. Your job is to find bugs and style issues.
Return a structured review with severity levels.
"""

agent, warn = load_agents_from_markdown(SAMPLE_AGENT_MD, "~/.easy-agent/agents/code-reviewer.md", "user")
print("Loaded agent :", agent.agentType if agent else warn)
if agent:
    print("  whenToUse  :", agent.whenToUse)
    print("  tools      :", agent.tools)
    print("  disallowed :", agent.disallowedTools)
    print("  model      :", agent.model)
    print("  maxTurns   :", agent.maxTurns)
    print("  system     :", agent.getSystemPrompt()[:60] + "…")

## 6. Agent Bootstrap

`src/agents/bootstrap.ts`

Single startup entry point. Loads built-ins first, then user + project custom agents. `set_agents()` populates the registry with last-writer-wins semantics: **project > user > built-in**.

In [ ]:
@dataclass
class AgentsBootstrapResult:
    builtInCount: int
    customCount: int
    warnings: list[str]

def bootstrap_agents(custom_agents: list[AgentDefinition] = None, custom_warnings: list[str] = None) -> AgentsBootstrapResult:
    """
    Merge built-ins + custom agents into the global registry.
    In the real TS: custom agents are loaded async from disk;
    here we accept them as a parameter for demonstration.
    """
    built_ins = get_built_in_agents()
    custom = custom_agents or []
    warnings = custom_warnings or []

    # Built-ins first, then custom layers on top → project overrides built-in
    REGISTRY.set_agents([*built_ins, *custom])

    return AgentsBootstrapResult(
        builtInCount=len(built_ins),
        customCount=len(custom),
        warnings=warnings,
    )

# Demo
result = bootstrap_agents(
    custom_agents=[agent] if agent else [],
    custom_warnings=[warn] if warn else [],
)
print("Bootstrap result:", result)
print("Registry contents:", [a.agentType for a in REGISTRY.get_all_agents()])
print("Find 'Explore':", REGISTRY.find_agent("Explore").agentType)
print("Find 'code-reviewer':", REGISTRY.find_agent("code-reviewer"))

## 7. Tool Resolution

`src/agents/resolveAgentTools.ts`

Builds the sub-agent's tool pool from the parent's pool. Algorithm (in order):
1. **Strip `Agent` tool** — structural guarantee: sub-agents cannot spawn sub-sub-agents
2. **Apply `disallowedTools`** — stripped even when `tools` is wildcard
3. **Wildcard check** — if `tools` is `None`/`[]`/`['*']`, keep everything remaining
4. **Intersect** with named `tools` list; collect unmatched names as `invalidTools`

In [ ]:
@dataclass
class ResolvedAgentTools:
    hasWildcard: bool
    resolvedTools: list[str]  # tool names (using strings here; real TS uses Tool objects)
    invalidTools: list[str]   # names in agent.tools that didn't match any parent tool

AGENT_TOOL_NAME = "Agent"

def resolve_agent_tools(
    agent_tools: Optional[list[str]],
    agent_disallowed: list[str],
    available_tools: list[str],
) -> ResolvedAgentTools:
    # Step 1: Strip the Agent tool (no sub-sub-agents)
    no_agent = [t for t in available_tools if t != AGENT_TOOL_NAME]

    # Step 2: Apply disallowedTools
    disallowed = set(agent_disallowed)
    after_disallow = [t for t in no_agent if t not in disallowed]

    # Step 3: Wildcard?
    has_wildcard = (
        agent_tools is None
        or len(agent_tools) == 0
        or (len(agent_tools) == 1 and agent_tools[0] == "*")
    )
    if has_wildcard:
        return ResolvedAgentTools(hasWildcard=True, resolvedTools=after_disallow, invalidTools=[])

    # Step 4: Intersect
    by_name = set(after_disallow)
    resolved = []
    seen = set()
    invalid = []
    for wanted in agent_tools:
        if wanted not in by_name:
            invalid.append(wanted)
        elif wanted not in seen:
            seen.add(wanted)
            resolved.append(wanted)

    return ResolvedAgentTools(hasWildcard=False, resolvedTools=resolved, invalidTools=invalid)

# Simulate parent's tool pool
PARENT_TOOLS = ["Read", "Edit", "Write", "Bash", "Grep", "Glob", "Agent", "MemoryWrite"]

# Explore agent (disallowedTools=[Write, Edit, MemoryWrite], tools=None → wildcard after disallow)
explore_resolved = resolve_agent_tools(
    agent_tools=EXPLORE_AGENT.tools,
    agent_disallowed=EXPLORE_AGENT.disallowedTools,
    available_tools=PARENT_TOOLS,
)
print("Explore resolved:", explore_resolved)

# general-purpose (wildcard)
gp_resolved = resolve_agent_tools(
    agent_tools=GENERAL_PURPOSE_AGENT.tools,
    agent_disallowed=GENERAL_PURPOSE_AGENT.disallowedTools,
    available_tools=PARENT_TOOLS,
)
print("General-purpose resolved:", gp_resolved)

# Custom reviewer with explicit allow-list (one invalid tool name for demo)
reviewer_tools = ["Read", "Grep", "Glob", "NonExistentTool"]
custom_resolved = resolve_agent_tools(
    agent_tools=reviewer_tools,
    agent_disallowed=["Write", "Edit"],
    available_tools=PARENT_TOOLS,
)
print("code-reviewer resolved:", custom_resolved)

## 8. Agent Prompt Injection (Discovery Listing)

`src/agents/promptInjection.ts`

Formats a `<system-reminder>` block listing available agents with their `whenToUse` descriptions. Injected into the dynamic zone of the system prompt so the model knows what `subagent_type` values to use.

- Built-ins sorted to top, then alphabetical by `agentType`
- Descriptions truncated to 220 chars
- Includes foreground vs background discipline rules and creation template

In [ ]:
MAX_DESC_CHARS = 220

def _truncate(s: str, max_len: int) -> str:
    if len(s) <= max_len:
        return s
    return s[:max_len - 1].rstrip() + "…"

CREATION_GUIDANCE = """
Defining a new sub-agent:
- File: <cwd>/.easy-agent/agents/<name>.md  (project) or  ~/.easy-agent/agents/<name>.md (user)
- Required frontmatter: name, description
- Optional: tools, disallowedTools, model, maxTurns, permissionMode
- Body below frontmatter IS the system prompt
"""

def format_agents_system_reminder(agents: list[AgentDefinition]) -> str:
    if not agents:
        return ""

    # Built-ins first, then alphabetical
    sorted_agents = sorted(
        agents,
        key=lambda a: (0 if a.source == "built-in" else 1, a.agentType.lower())
    )

    lines = sorted_agents
    desc_lines = [
        f"- {a.agentType} [{a.source}]: {_truncate(a.whenToUse, MAX_DESC_CHARS)}"
        for a in sorted_agents
    ]

    return "\n".join([
        "<system-reminder>",
        "Available sub-agents you can invoke via the `Agent` tool.",
        "Call Agent(prompt='...', subagent_type='<name>') to delegate a subtask.",
        "",
        "Foreground vs background:",
        "- Foreground (default): parent blocks until sub-agent completes.",
        "- Background (run_in_background=true): fire-and-forget; notified via <task-notification>.",
        "",
        *desc_lines,
        CREATION_GUIDANCE,
        "</system-reminder>",
    ])

reminder = format_agents_system_reminder(REGISTRY.get_all_agents())
print(reminder)

## 9. System Prompt Assembly

`src/context/systemPrompt.ts`

The system prompt is divided into two zones:

| Zone | Tag | Contents |
|---|---|---|
| **Static** | `<SYSTEM_STATIC_CONTEXT>` | Core behavioral instructions (stable, cached) |
| **Dynamic** | `<SYSTEM_DYNAMIC_CONTEXT>` | Env context, AGENT.md, memory, skills, agents |

Three sources are fetched in **parallel**: runtime environment (git status), AGENT.md context, and the memory entrypoint.

> **Note:** Real LLM calls and filesystem reads are omitted; this cell shows the composition logic.

In [ ]:
SYSTEM_PROMPT_STATIC_START = "<SYSTEM_STATIC_CONTEXT>"
SYSTEM_PROMPT_STATIC_END = "</SYSTEM_STATIC_CONTEXT>"
SYSTEM_PROMPT_DYNAMIC_START = "<SYSTEM_DYNAMIC_CONTEXT>"
SYSTEM_PROMPT_DYNAMIC_END = "</SYSTEM_DYNAMIC_CONTEXT>"

@dataclass
class RuntimeEnvironmentContext:
    cwd: str
    date: str
    os_info: str
    gitBranch: Optional[str] = None
    gitStatus: Optional[str] = None
    gitRecentCommit: Optional[str] = None

def get_static_prompt_sections() -> list[str]:
    return [
        "You are Easy Agent, a terminal-native local coding assistant.",
        "Operate directly, be concise, prefer concrete actions with tools.",
        "Prefer specialized tools over shell: Read, Edit, Write, Grep, Glob, Bash.",
        "Treat the current working directory as the primary workspace boundary.",
    ]

def format_environment_context(ctx: RuntimeEnvironmentContext) -> str:
    lines = [
        "Environment:",
        f"- Current working directory: {ctx.cwd}",
        f"- Current date: {ctx.date}",
        f"- Operating system: {ctx.os_info}",
    ]
    if ctx.gitBranch:
        lines.append(f"- Git branch: {ctx.gitBranch}")
    if ctx.gitStatus:
        lines.append(f"- Git status snapshot:\n{ctx.gitStatus}")
    if ctx.gitRecentCommit:
        lines.append(f"- Recent commit: {ctx.gitRecentCommit}")
    return "\n".join(lines)

def build_system_prompt(
    cwd: str,
    env_ctx: RuntimeEnvironmentContext,
    agent_md: Optional[str] = None,
    memory_entrypoint: Optional[str] = None,
    additional_instructions: Optional[str] = None,
    agents: Optional[list[AgentDefinition]] = None,
) -> list[str]:
    """
    Compose the full system prompt sections list.
    In production, env_ctx, agent_md, and memory_entrypoint are fetched
    in parallel with asyncio.gather().
    """
    static_sections = [
        SYSTEM_PROMPT_STATIC_START,
        *get_static_prompt_sections(),
        SYSTEM_PROMPT_STATIC_END,
    ]

    memory_sections = []
    if memory_entrypoint:
        memory_sections.append(f"Memory index:\n{memory_entrypoint}")

    agents_reminder = format_agents_system_reminder(agents or [])

    dynamic_sections = [
        s for s in [
            SYSTEM_PROMPT_DYNAMIC_START,
            format_environment_context(env_ctx),
            f"Project memory (AGENT.md):\n{agent_md}" if agent_md else "",
            "\n\n".join(memory_sections) if memory_sections else "",
            f"Session instructions:\n{additional_instructions}" if additional_instructions else "",
            agents_reminder,
            SYSTEM_PROMPT_DYNAMIC_END,
        ] if s
    ]

    return [*static_sections, *dynamic_sections]

def render_system_prompt(parts: list[str]) -> str:
    return "\n\n".join(parts)

# Demo
demo_env = RuntimeEnvironmentContext(
    cwd=str(PROJECT_ROOT),
    date=datetime.now(timezone.utc).isoformat(),
    os_info="darwin 25.4.0 (arm64)",
    gitBranch="main",
    gitStatus="M src/agents/registry.ts",
    gitRecentCommit="a1b2c3d feat: add async agent store",
)
parts = build_system_prompt(
    cwd=str(PROJECT_ROOT),
    env_ctx=demo_env,
    agent_md="## Rules\nPrefer concise responses.",
    memory_entrypoint="- [API conventions](api-conventions.md) — REST patterns",
    agents=REGISTRY.get_all_agents(),
)
full_prompt = render_system_prompt(parts)
print(f"Total sections : {len(parts)}")
print(f"Prompt length  : {len(full_prompt)} chars")
print("--- First 600 chars ---")
print(full_prompt[:600])

## 10. AGENT.md Hierarchical Loading

`src/context/claudeMd.ts`

AGENT.md files provide project-specific instructions. They are loaded from the **global config** and every directory in the cwd chain (root → leaf), so more-specific overrides come last.

```
/AGENT.md
/Users/AGENT.md
/Users/nick/AGENT.md
/Users/nick/Code/AGENT.md
/Users/nick/Code/easy-agent-main/AGENT.md    ← most specific
```

In [ ]:
def strip_html_comments(content: str) -> str:
    return re.sub(r'<!--[\s\S]*?-->', '', content).strip()

def get_directory_chain(cwd: str) -> list[str]:
    """Root → leaf chain of directories from filesystem root to cwd."""
    resolved = Path(cwd).resolve()
    chain = []
    current = resolved
    while True:
        chain.append(str(current))
        parent = current.parent
        if parent == current:
            break
        current = parent
    chain.reverse()   # root first
    return chain

def get_agent_md_files(cwd: str) -> list[str]:
    global_path = str(Path.home() / ".easy-agent" / "AGENT.md")
    files = [global_path]
    for d in get_directory_chain(cwd):
        files.append(str(Path(d) / "AGENT.md"))
    return files

def load_agent_md_context(cwd: str) -> str:
    """
    Load all AGENT.md files in the hierarchy.
    (Simplified: reads from disk if files exist, otherwise returns empty.)
    """
    sections = []
    for file_path in get_agent_md_files(cwd):
        p = Path(file_path)
        if p.exists() and p.is_file():
            raw = p.read_text("utf-8")
            stripped = strip_html_comments(raw).strip()
            if stripped:
                sections.append(f"# Source: {file_path}\n{stripped}")
    return "\n\n".join(sections)

chain = get_directory_chain(str(PROJECT_ROOT))
print("Directory chain (last 4):", chain[-4:])
print("AGENT.md candidates:")
for f in get_agent_md_files(str(PROJECT_ROOT))[-4:]:
    exists = Path(f).exists()
    print(f"  {'✓' if exists else '·'} {f}")

## 11. Memory System

`src/context/memory/memdir.ts`

Persistent project knowledge stored as markdown files with YAML frontmatter.

**Storage layout:**
```
~/.easy-agent/projects/<projectKey>/memory/
  MEMORY.md               ← entrypoint index (≤200 lines, ≤25 KB)
  api-conventions.md      ← individual memory file
  feedback-testing.md
```

**Project identity** = `{sanitized-basename}-{sha256-prefix-16}` of the git root path.

**Fuzzy deduplication:** before creating a new file, checks for an exact or substring name match.

In [ ]:
MEMORY_ENTRYPOINT = "MEMORY.md"
MAX_ENTRYPOINT_LINES = 200
MAX_ENTRYPOINT_BYTES = 25_000

def sanitize_slug(s: str) -> str:
    slug = re.sub(r'[^a-z0-9._-]+', '-', s.lower())
    slug = slug.strip('-')[:80]
    return slug or "project"

def get_project_key(git_root: str) -> str:
    """Stable per-project identifier derived from git root path."""
    base = sanitize_slug(Path(git_root).name)
    suffix = hashlib.sha256(git_root.encode()).hexdigest()[:16]
    return f"{base}-{suffix}"

def get_project_memory_dir(git_root: str) -> Path:
    key = get_project_key(git_root)
    return EASY_AGENT_HOME / "projects" / key / "memory"

def parse_memory_frontmatter(raw: str) -> Optional[dict]:
    m = re.match(r'^---\n([\s\S]*?)\n---\n?', raw)
    if not m:
        return None
    fields = {}
    for line in m.group(1).splitlines():
        if ':' in line:
            k, _, v = line.partition(':')
            fields[k.strip()] = v.strip()
    required = {"name", "description", "type"}
    if not required.issubset(fields):
        return None
    return fields

def slugify_memory_filename(name: str) -> str:
    return sanitize_slug(name).replace('.', '-') + ".md"

def build_pointer_line(filename: str, title: str, hook: str) -> str:
    return f"- [{title}]({filename}) — {hook}"

def truncate_entrypoint(raw: str) -> str:
    lines = raw.splitlines()
    if len(lines) > MAX_ENTRYPOINT_LINES:
        lines = lines[:MAX_ENTRYPOINT_LINES]
        lines.append("> WARNING: MEMORY.md was truncated by line limit.")
    content = "\n".join(lines)
    if len(content.encode('utf-8')) > MAX_ENTRYPOINT_BYTES:
        content = content.encode('utf-8')[:MAX_ENTRYPOINT_BYTES].decode('utf-8', errors='ignore')
    return content

# Demo: compute project identity for this repo
git_root = str(PROJECT_ROOT)
project_key = get_project_key(git_root)
memory_dir = get_project_memory_dir(git_root)
print(f"git_root    : {git_root}")
print(f"project_key : {project_key}")
print(f"memory_dir  : {memory_dir}")

# Demo: construct a memory document body
def format_memory_document(name: str, description: str, mem_type: str, content: str) -> str:
    return "\n".join([
        "---",
        f"name: {name}",
        f"description: {description}",
        f"type: {mem_type}",
        "---",
        "",
        content.strip(),
        "",
    ])

sample_memory = format_memory_document(
    name="api-conventions",
    description="REST API patterns used in this codebase",
    mem_type="project",
    content="All endpoints use camelCase. Auth via Bearer tokens. Pagination via cursor.",
)
print("\nSample memory file content:")
print(sample_memory)

## 12. Two-Tier Compaction

`src/context/compaction.ts`

Manages conversation length via two tiers:

| Tier | Cost | Mechanism |
|---|---|---|
| **Micro-compaction** | Cheap (no API call) | Replace old tool results with placeholder text |
| **Full compaction** | Expensive (API call) | LLM summarizes entire conversation; preserves last 8 messages verbatim |

**Micro-compaction** rules:
- Only targets messages in `COMPACTABLE_TOOLS` = {Read, Grep, Glob, Bash, Edit, Write}
- Skips the most recent 8 messages (`MICROCOMPACT_KEEP_RECENT_MESSAGES`)
- Minimum 10 messages before it activates (`MICROCOMPACT_MIN_MESSAGES`)
- Binary-only content (images) → `"[image]"` placeholder

**Full compaction** sequence:
1. Run micro-compaction first
2. Summarize with LLM (using `NO_TOOLS_PREAMBLE` to prevent tool calls during summary)
3. Find safe tail boundary (no dangling tool_result/tool_use pairs)
4. Compose: `[user summary message, CompactBoundary marker, ...tail]`

> **Note:** The `summarize_messages()` call (actual LLM API) is stubbed below with a placeholder.

In [ ]:
OLD_TOOL_RESULT_PLACEHOLDER = "[Old tool result content cleared]"
MICROCOMPACT_MIN_MESSAGES = 10
MICROCOMPACT_KEEP_RECENT_MESSAGES = 8
COMPACTABLE_TOOLS = {"Read", "Grep", "Glob", "Bash", "Edit", "Write"}

# ── Message type helpers ──────────────────────────────────────────────────

def make_user_msg(content: str) -> dict:
    return {"role": "user", "content": content}

def make_assistant_msg(content) -> dict:
    return {"role": "assistant", "content": content}

def make_tool_use_block(tool_id: str, name: str, inp: dict) -> dict:
    return {"type": "tool_use", "id": tool_id, "name": name, "input": inp}

def make_tool_result_block(tool_use_id: str, content: str) -> dict:
    return {"type": "tool_result", "tool_use_id": tool_use_id, "content": content}

def collect_tool_ids(msg: dict) -> list[str]:
    content = msg.get("content", [])
    if not isinstance(content, list): return []
    return [b["id"] for b in content if isinstance(b, dict) and b.get("type") == "tool_use"]

def collect_tool_result_ids(msg: dict) -> list[str]:
    content = msg.get("content", [])
    if not isinstance(content, list): return []
    return [b["tool_use_id"] for b in content if isinstance(b, dict) and b.get("type") == "tool_result"]

# ── Micro-compaction ─────────────────────────────────────────────────────

def micro_compact_message(msg: dict) -> tuple[dict, list[str]]:
    """Replace old tool results in a single message. Returns (new_msg, compacted_ids)."""
    content = msg.get("content", [])
    if not isinstance(content, list):
        return msg, []

    new_content = []
    compacted_ids = []
    for block in content:
        if not isinstance(block, dict) or block.get("type") != "tool_result":
            new_content.append(block)
            continue

        block_content = block.get("content", "")
        # Binary-only → [image]
        if isinstance(block_content, list) and all(
            b.get("type") in ("image", "document") for b in block_content if isinstance(b, dict)
        ):
            compacted_ids.append(block["tool_use_id"])
            new_content.append({**block, "content": "[image]"})
            continue

        # Text tool result — check tool name prefix
        if not isinstance(block_content, str):
            new_content.append(block)
            continue
        m = re.match(r'^([A-Za-z0-9_-]+):', block_content)
        tool_name = m.group(1) if m else None
        if tool_name and tool_name in COMPACTABLE_TOOLS:
            compacted_ids.append(block["tool_use_id"])
            new_content.append({**block, "content": OLD_TOOL_RESULT_PLACEHOLDER})
        else:
            new_content.append(block)

    return {**msg, "content": new_content}, compacted_ids

def micro_compact_messages(messages: list[dict]) -> tuple[list[dict], list[str]]:
    """Apply micro-compaction to all messages except the most recent 8."""
    if len(messages) < MICROCOMPACT_MIN_MESSAGES:
        return messages, []

    all_compacted_ids = []
    result = []
    for i, msg in enumerate(messages):
        if i >= len(messages) - MICROCOMPACT_KEEP_RECENT_MESSAGES:
            result.append(msg)
        else:
            new_msg, ids = micro_compact_message(msg)
            all_compacted_ids.extend(ids)
            result.append(new_msg)
    return result, all_compacted_ids

# ── Full compaction helpers ───────────────────────────────────────────────

def make_compact_boundary(compact_type: str, original_count: int, reason: str = "") -> dict:
    parts = ["[CompactBoundary]", f"type={compact_type}", f"messages={original_count}"]
    if reason:
        parts.append(f"reason={reason}")
    return {"role": "assistant", "content": " ".join(parts)}

def find_preserved_tail_start(messages: list[dict], desired_count: int = 8) -> int:
    """Find the earliest index for a clean tail (no dangling tool_result without its tool_use)."""
    start = max(0, len(messages) - desired_count)
    while start > 0:
        tail = messages[start:]
        tool_uses = set(id_ for m in tail for id_ in collect_tool_ids(m))
        tool_results = set(id_ for m in tail for id_ in collect_tool_result_ids(m))
        dangling = [r for r in tool_results if r not in tool_uses]
        if not dangling:
            return start
        start -= 1
    return 0

# ── Demo ──────────────────────────────────────────────────────────────────

# Build a synthetic 12-message conversation with tool calls
synthetic_messages = []
for i in range(5):
    tool_id = f"tool_{i}"
    synthetic_messages.append(make_user_msg(f"User question {i}"))
    synthetic_messages.append(make_assistant_msg([
        make_tool_use_block(tool_id, "Read", {"file_path": f"/src/file{i}.ts"}),
    ]))
    synthetic_messages.append(make_user_msg([
        make_tool_result_block(tool_id, f"Read: content of file{i}.ts — lots of text here " * 5),
    ]))

synthetic_messages.append(make_user_msg("Final user question"))
synthetic_messages.append(make_assistant_msg("Final answer"))

print(f"Original message count: {len(synthetic_messages)}")

# Apply micro-compaction
micro_compacted, compacted_ids = micro_compact_messages(synthetic_messages)
compacted_count = sum(
    1 for m in micro_compacted
    for b in (m.get("content") if isinstance(m.get("content"), list) else [])
    if isinstance(b, dict) and b.get("content") == OLD_TOOL_RESULT_PLACEHOLDER
)
print(f"After micro-compact   : {len(micro_compacted)} msgs, {len(compacted_ids)} tool results replaced")

# Show the tail boundary
tail_start = find_preserved_tail_start(micro_compacted, desired_count=8)
print(f"Preserved tail starts at index: {tail_start} (preserves {len(micro_compacted)-tail_start} msgs)")

# Full compaction (stubbed: real code calls LLM API)
def summarize_messages_stub(messages: list[dict]) -> str:
    """Stub for the real LLM summarization API call."""
    return (
        "<analysis>\nConversation covered 5 file reads.\n</analysis>\n\n"
        "<summary>\n"
        "1. Primary Request: Read 5 source files to understand the codebase.\n"
        "2. Key Files: /src/file0.ts through /src/file4.ts\n"
        "3. Current Work: Answering final user question.\n"
        "</summary>"
    )

summary = summarize_messages_stub(micro_compacted)
tail = micro_compacted[tail_start:]

compacted_messages = [
    make_user_msg(
        f"This session is being continued from a previous conversation. "
        f"Summary:\n\n{summary}"
    ),
    make_compact_boundary("auto", len(micro_compacted)),
    *tail,
]
print(f"\nAfter full compact: {len(compacted_messages)} msgs ({len(tail)} tail + 2 header)")
print("CompactBoundary msg:", compacted_messages[1]["content"])

## 13. Auto-Compaction (Circuit Breaker)

`src/context/autoCompact.ts`

Triggers compaction automatically when token usage approaches the context window limit.

**Token warning states:**

| State | Threshold | Action |
|---|---|---|
| `normal` | < warning threshold | No action |
| `warning` | ≥ 80% effective window | UI indicator |
| `error` | ≥ auto-compact threshold | Triggers compaction |
| `blocking` | ≥ blocking limit | Blocks new submissions |

**Circuit breaker:** After 3 consecutive failures (`MAX_CONSECUTIVE_AUTOCOMPACT_FAILURES`), stops attempting auto-compact.
**Recursion guard:** Skips compaction when `querySource` is `"compact"` or `"session_memory"`.

In [ ]:
MAX_CONSECUTIVE_AUTOCOMPACT_FAILURES = 3
TokenWarningState = Literal["normal", "warning", "error", "blocking"]

@dataclass
class TokenWarningResult:
    state: TokenWarningState
    estimatedTokens: int
    threshold: int        # auto-compact threshold
    blockingLimit: int
    contextWindow: int

# Simplified token budget (real code uses tiktoken-like estimation)
MODEL_CONTEXT_WINDOWS = {
    "claude-opus-4-7": 200_000,
    "claude-sonnet-4-6": 200_000,
    "claude-haiku-4-5-20251001": 200_000,
}
AUTOCOMPACT_BUFFER_TOKENS   = 20_000  # compact when within 20K of limit
WARNING_THRESHOLD_BUFFER     = 40_000
MANUAL_COMPACT_BUFFER        = 10_000

def get_context_window(model: str) -> int:
    return MODEL_CONTEXT_WINDOWS.get(model, 200_000)

def get_effective_window(model: str) -> int:
    return get_context_window(model)  # simplified; real code subtracts system-prompt budget

def get_auto_compact_threshold(model: str) -> int:
    return max(0, get_effective_window(model) - AUTOCOMPACT_BUFFER_TOKENS)

def get_blocking_limit(model: str) -> int:
    return max(0, get_effective_window(model) - MANUAL_COMPACT_BUFFER)

def calculate_token_warning_state(estimated_tokens: int, model: str) -> TokenWarningResult:
    ctx = get_context_window(model)
    effective = get_effective_window(model)
    blocking = get_blocking_limit(model)
    threshold = get_auto_compact_threshold(model)
    warning_thresh = max(0, effective - WARNING_THRESHOLD_BUFFER)

    if estimated_tokens >= blocking:
        state: TokenWarningState = "blocking"
    elif estimated_tokens >= threshold:
        state = "error"
    elif estimated_tokens >= warning_thresh:
        state = "warning"
    else:
        state = "normal"

    return TokenWarningResult(
        state=state, estimatedTokens=estimated_tokens,
        threshold=threshold, blockingLimit=blocking, contextWindow=ctx,
    )

# Module-level failure counter (mirrors TS module state)
_consecutive_failures = 0

def reset_auto_compact_failures():
    global _consecutive_failures
    _consecutive_failures = 0

def should_auto_compact(estimated_tokens: int, model: str, query_source: Optional[str] = None) -> bool:
    global _consecutive_failures
    # Recursion guards
    if query_source in ("compact", "session_memory"):
        return False
    # Circuit breaker
    if _consecutive_failures >= MAX_CONSECUTIVE_AUTOCOMPACT_FAILURES:
        return False
    return estimated_tokens >= get_auto_compact_threshold(model)

# Demo
model = "claude-sonnet-4-6"
for tokens in [150_000, 175_000, 185_000, 195_000]:
    result = calculate_token_warning_state(tokens, model)
    should = should_auto_compact(tokens, model)
    print(f"  {tokens:>7,} tokens → state={result.state:<8}  should_compact={should}")

## 14. Plan Mode

`src/context/plans.ts` + `src/context/planAttachments.ts`

**Plan files** are written to `~/.easy-agent/plans/<8-hex-slug>.md`. The slug is cached for the session lifetime.

**Plan attachments** are injected as user messages (not system prompt text), throttled to avoid overwhelming the context:

| Condition | Attachment type |
|---|---|
| First time in plan mode | Full instructions |
| Every `TURNS_BETWEEN_ATTACHMENTS` (5) human turns | Full reminder |
| Other turns | Sparse (brief) reminder |
| Exiting plan mode | One-shot exit message |

In [ ]:
import secrets

PLAN_ATTACHMENT_MARKER = "[plan_mode_attachment]"
PLAN_EXIT_MARKER = "[plan_mode_exit]"
TURNS_BETWEEN_ATTACHMENTS = 5
FULL_REMINDER_EVERY_N = 5

_cached_plan_slug: Optional[str] = None

def get_plan_slug() -> str:
    global _cached_plan_slug
    if not _cached_plan_slug:
        _cached_plan_slug = secrets.token_hex(4)
    return _cached_plan_slug

def reset_plan_slug():
    global _cached_plan_slug
    _cached_plan_slug = None

def get_plan_file_path() -> str:
    plans_root = EASY_AGENT_HOME / "plans"
    return str(plans_root / f"{get_plan_slug()}.md")

def build_full_plan_mode_text(plan_file_path: str) -> str:
    return "\n".join([
        PLAN_ATTACHMENT_MARKER, "",
        "PLAN MODE ACTIVE — You are currently in plan mode.", "",
        "Workflow:",
        "1. EXPLORE: Use Read, Grep, Glob, read-only Bash.",
        "2. PLAN: Write a detailed implementation plan.",
        "3. EXIT: Call ExitPlanMode when ready.", "",
        "Rules:",
        "- Do NOT use Edit or destructive Bash.",
        "- Write ONLY to the plan file below.",
        "- You MUST end your turn by exploring or calling ExitPlanMode.", "",
        f"Plan file: {plan_file_path}",
    ])

def build_sparse_plan_mode_text(plan_file_path: str) -> str:
    return "\n".join([
        PLAN_ATTACHMENT_MARKER, "",
        "Reminder: You are still in PLAN MODE. Only read-only tools are allowed.",
        f"Write your plan to: {plan_file_path}",
        "Call ExitPlanMode when your plan is ready.",
    ])

def build_plan_mode_exit_text(plan_file_path: str, plan_exists: bool) -> str:
    lines = [PLAN_EXIT_MARKER, "", "You have exited plan mode. Full tool access is now restored."]
    if plan_exists:
        lines += [f"Your approved plan is at: {plan_file_path}",
                  "Proceed with implementing the plan."]
    return "\n".join(lines)

def count_human_turns_since_last_attachment(messages: list[dict]) -> int:
    count = 0
    for m in reversed(messages):
        if m["role"] != "user": continue
        if isinstance(m["content"], str) and (PLAN_ATTACHMENT_MARKER in m["content"] or PLAN_EXIT_MARKER in m["content"]):
            return count
        if isinstance(m["content"], str):
            count += 1
    return count

def get_plan_mode_attachment(messages: list[dict], plan_file_path: str) -> Optional[dict]:
    has_any = any(
        m["role"] == "user" and isinstance(m["content"], str) and PLAN_ATTACHMENT_MARKER in m["content"]
        for m in messages
    )
    if not has_any:
        return {"role": "user", "content": build_full_plan_mode_text(plan_file_path)}

    turns_since = count_human_turns_since_last_attachment(messages)
    if turns_since < TURNS_BETWEEN_ATTACHMENTS:
        return None  # throttled

    # Count attachments to decide full vs sparse
    attach_count = sum(
        1 for m in messages
        if m["role"] == "user" and isinstance(m["content"], str) and PLAN_ATTACHMENT_MARKER in m["content"]
    )
    is_full = (attach_count + 1) % FULL_REMINDER_EVERY_N == 1
    text = build_full_plan_mode_text(plan_file_path) if is_full else build_sparse_plan_mode_text(plan_file_path)
    return {"role": "user", "content": text}

# Demo
reset_plan_slug()
plan_path = get_plan_file_path()
print("Plan file path:", plan_path)

# Simulate entering plan mode
messages_plan = []
attachment = get_plan_mode_attachment(messages_plan, plan_path)
print("\n-- First attachment (full) --")
print(attachment["content"])

# Simulate 6 human turns later
messages_plan.append(attachment)
for i in range(6):
    messages_plan.append({"role": "user", "content": f"Human turn {i}"})
    messages_plan.append({"role": "assistant", "content": f"Assistant reply {i}"})

next_att = get_plan_mode_attachment(messages_plan, plan_path)
print("\n-- After 6 human turns --")
print(next_att["content"] if next_att else "throttled — None returned")

## 15. Session Persistence

`src/session/storage.ts`

Sessions are stored as **JSONL files** (one JSON object per line). Each line is a `TranscriptEntry` with a `type` discriminator.

```
~/.easy-agent/projects/<projectKey>/
  latest                ← most recent session ID
  <sessionId>.jsonl     ← full transcript
```

**Transcript entry types:**

| type | Contents |
|---|---|
| `session_meta` | session ID, cwd, model, startedAt |
| `message` | role + MessageParam |
| `tool_event` | tool name, phase (start/done), result length |
| `usage` | per-turn + cumulative token counts |
| `system` | info/error system messages |
| `compaction` | auto/manual compaction marker |

**Session restore** algorithm:
1. Read all JSONL entries
2. Find the **last compaction marker**
3. Extract only messages *after* that marker (pre-compaction context is already summarized)

In [ ]:
@dataclass
class SessionMetadata:
    sessionId: str
    cwd: str
    startedAt: str
    model: str

@dataclass
class SessionSummary:
    sessionId: str
    cwd: str
    startedAt: str
    updatedAt: str
    model: str
    messageCount: int
    totalInputTokens: int
    totalOutputTokens: int

def create_session_id() -> str:
    return str(uuid.uuid4())

def get_session_paths(cwd: str, session_id: str) -> dict:
    project_key = get_project_key(cwd)
    project_dir = EASY_AGENT_HOME / "projects" / project_key
    return {
        "projectDir": project_dir,
        "transcriptPath": project_dir / f"{session_id}.jsonl",
        "latestPath": project_dir / "latest",
    }

def parse_jsonl_line(line: str) -> Optional[dict]:
    """Parse one JSONL line; return None on error."""
    try:
        obj = json.loads(line.strip())
        return obj if isinstance(obj, dict) else None
    except json.JSONDecodeError:
        return None

def build_transcript_entry(entry_type: str, **kwargs) -> str:
    """Serialize a transcript entry to a JSONL line."""
    obj = {"type": entry_type, "timestamp": datetime.now(timezone.utc).isoformat(), **kwargs}
    return json.dumps(obj)

# Demo: simulate writing and restoring a session
def simulate_session():
    sid = create_session_id()
    lines = []

    # session_meta (no timestamp)
    lines.append(json.dumps({
        "type": "session_meta",
        "sessionId": sid,
        "cwd": str(PROJECT_ROOT),
        "startedAt": datetime.now(timezone.utc).isoformat(),
        "model": "claude-sonnet-4-6",
    }))
    # Some messages
    lines.append(build_transcript_entry("message", role="user",
        message={"role": "user", "content": "Hello"}))
    lines.append(build_transcript_entry("message", role="assistant",
        message={"role": "assistant", "content": "Hi! How can I help?"}))
    # usage
    lines.append(build_transcript_entry("usage",
        turn={"input_tokens": 100, "output_tokens": 50},
        total={"input_tokens": 100, "output_tokens": 50}))
    # compaction marker (simulate a compaction happening)
    lines.append(build_transcript_entry("compaction", trigger="auto"))
    # messages after compaction (these are what get restored)
    lines.append(build_transcript_entry("message", role="user",
        message={"role": "user", "content": "What about X?"}))
    lines.append(build_transcript_entry("message", role="assistant",
        message={"role": "assistant", "content": "X does Y."}))

    return "\n".join(lines), sid

def restore_session_from_jsonl(raw_jsonl: str) -> dict:
    """
    Restore session: parse JSONL, find last compaction marker,
    return only messages after it.
    """
    entries = [parse_jsonl_line(l) for l in raw_jsonl.splitlines() if l.strip()]
    entries = [e for e in entries if e is not None]

    meta = next((e for e in entries if e.get("type") == "session_meta"), None)
    if not meta:
        raise ValueError("Missing session_meta")

    # Find last compaction marker
    start_index = 0
    for i, e in enumerate(entries):
        if e.get("type") == "compaction":
            start_index = i + 1

    messages = [
        e["message"] for e in entries[start_index:]
        if e.get("type") == "message"
    ]

    latest_usage = next(
        (e for e in reversed(entries) if e.get("type") == "usage"), None
    )

    return {
        "sessionId": meta["sessionId"],
        "cwd": meta["cwd"],
        "model": meta["model"],
        "messageCount": len(messages),
        "messages": messages,
        "totalTokens": (latest_usage["total"] if latest_usage else {}),
    }

raw, sid = simulate_session()
print("--- Simulated JSONL transcript ---")
for line in raw.splitlines():
    e = json.loads(line)
    print(f"  [{e['type']}]", end=" ")
    if e['type'] == 'message': print(f"role={e['role']}, content={str(e['message']['content'])[:40]}")
    elif e['type'] == 'usage': print(f"total={e['total']}")
    elif e['type'] == 'compaction': print(f"trigger={e['trigger']}")
    else: print()

print("\n--- Restored session ---")
restored = restore_session_from_jsonl(raw)
print(f"  sessionId    : {restored['sessionId'][:8]}…")
print(f"  model        : {restored['model']}")
print(f"  messageCount : {restored['messageCount']}  (only messages AFTER compaction marker)")
print(f"  messages     : {[m['content'] for m in restored['messages']]}")

## 16. Async Agent Store (Pub/Sub Pattern)

`src/state/asyncAgentStore.ts`

Manages the in-memory state of background sub-agents. Follows the **module-level pub/sub pattern** used by all state stores:

- Module-level `dict` for state
- `set[Listener]` for subscribers
- `subscribe()` returns an **unsubscribe** function
- Mutation functions call `notify()` after updating
- Listeners are try-caught so UI errors cannot break mutations

**Lifecycle:** `register → [updateProgress]* → complete | fail | kill`

> The parent's `AbortController` is **not linked** to the background agent — pressing ESC on the parent loop does NOT interrupt a backgrounded sub-agent.

In [ ]:
AsyncAgentStatus = Literal["running", "completed", "failed", "killed"]

@dataclass
class AsyncAgentEntry:
    agentId: str
    agentType: str
    prompt: str
    startedAt: str
    status: AsyncAgentStatus
    outputFile: str
    isolated: bool
    toolUseCount: int = 0
    description: Optional[str] = None
    lastToolName: Optional[str] = None
    totalTokens: Optional[int] = None
    inputTokens: Optional[int] = None
    outputTokens: Optional[int] = None
    turnCount: Optional[int] = None
    finalText: Optional[str] = None
    error: Optional[str] = None
    durationMs: Optional[int] = None
    reason: Optional[str] = None
    worktreePath: Optional[str] = None
    worktreeBranch: Optional[str] = None

# Module-level state (mirrors TS module scope)
_async_agent_entries: dict[str, AsyncAgentEntry] = {}
_async_agent_listeners: set[Callable] = set()

def _notify_async(agent_id: str, snapshot: Optional[AsyncAgentEntry]) -> None:
    for l in _async_agent_listeners:
        try: l(agent_id, snapshot)
        except: pass

def subscribe_async_agents(listener: Callable) -> Callable:
    _async_agent_listeners.add(listener)
    return lambda: _async_agent_listeners.discard(listener)

def register_async_agent(agent_id: str, agent_type: str, prompt: str,
                          output_file: str, description: Optional[str] = None,
                          isolated: bool = False) -> AsyncAgentEntry:
    if agent_id in _async_agent_entries:
        raise ValueError(f"Agent '{agent_id}' already registered")
    entry = AsyncAgentEntry(
        agentId=agent_id, agentType=agent_type, prompt=prompt,
        startedAt=datetime.now(timezone.utc).isoformat(),
        status="running", outputFile=output_file, isolated=isolated,
        description=description,
    )
    _async_agent_entries[agent_id] = entry
    _notify_async(agent_id, entry)
    return entry

def update_async_agent_progress(agent_id: str, **kwargs) -> None:
    cur = _async_agent_entries.get(agent_id)
    if not cur or cur.status != "running": return
    for k, v in kwargs.items():
        if hasattr(cur, k): setattr(cur, k, v)
    _notify_async(agent_id, cur)

def complete_async_agent(agent_id: str, result: AgentRunResult) -> None:
    cur = _async_agent_entries.get(agent_id)
    if not cur: return
    cur.status = "completed"
    cur.finalText = result.finalText
    cur.durationMs = result.totalDurationMs
    cur.totalTokens = result.totalTokens
    cur.inputTokens = result.inputTokens
    cur.outputTokens = result.outputTokens
    cur.toolUseCount = result.totalToolUseCount
    cur.turnCount = result.turnCount
    cur.reason = result.reason
    _notify_async(agent_id, cur)

def fail_async_agent(agent_id: str, error: str, duration_ms: int) -> None:
    cur = _async_agent_entries.get(agent_id)
    if not cur: return
    cur.status = "failed"
    cur.error = error
    cur.durationMs = duration_ms
    cur.reason = "model_error"
    _notify_async(agent_id, cur)

def kill_async_agent(agent_id: str) -> bool:
    cur = _async_agent_entries.get(agent_id)
    if not cur or cur.status != "running": return False
    cur.status = "killed"
    cur.reason = "aborted"
    _notify_async(agent_id, cur)
    return True

def get_running_async_agents() -> list[AsyncAgentEntry]:
    return [e for e in _async_agent_entries.values() if e.status == "running"]

# Demo
events_log = []
unsub = subscribe_async_agents(lambda aid, snap: events_log.append(
    f"[{snap.status if snap else 'removed'}] {aid[:8]}"
))

a_id = "agent-abc123"
entry = register_async_agent(a_id, "Explore", "Search for all usages of resolveAgentTools",
                               output_file=f"/tmp/{a_id}.jsonl", description="search task")
update_async_agent_progress(a_id, toolUseCount=1, lastToolName="Grep")
update_async_agent_progress(a_id, toolUseCount=2, lastToolName="Read")

mock_result = AgentRunResult(
    agentType="Explore", finalText="Found 3 usages in src/agents/",
    messages=[], totalToolUseCount=2, totalDurationMs=3200,
    totalTokens=4500, inputTokens=4000, outputTokens=500,
    turnCount=2, reason="completed",
)
complete_async_agent(a_id, mock_result)

print("Events fired:", events_log)
final = _async_agent_entries[a_id]
print(f"Final status : {final.status}")
print(f"Final text   : {final.finalText}")
print(f"Duration     : {final.durationMs} ms, tokens: {final.totalTokens}")

unsub()  # clean up subscription

## 17. Notification Store & Task Notifications

`src/state/notificationStore.ts`

Background sub-agents finish at unpredictable times. A FIFO queue holds `<task-notification>` messages that are **drained at the start of the next user submission** (not during an active stream).

**Why queued injection?** Injecting mid-stream would race with the API response. The QueryEngine's `submitInternal` drains the queue once before each new turn.

The `formatTaskNotification()` function builds the XML block the LLM will see.

In [ ]:
@dataclass
class PendingNotification:
    mode: str   # "task-notification"
    text: str
    enqueuedAt: float

_notification_queue: list[PendingNotification] = []
_notification_listeners: set[Callable] = set()

def subscribe_pending_notifications(listener: Callable) -> Callable:
    _notification_listeners.add(listener)
    return lambda: _notification_listeners.discard(listener)

def enqueue_pending_notification(mode: str, text: str) -> None:
    _notification_queue.append(PendingNotification(mode=mode, text=text, enqueuedAt=time.time()))
    for l in _notification_listeners:
        try: l()
        except: pass

def drain_pending_notifications() -> list[PendingNotification]:
    """Atomically take all queued notifications. Must inject all of them — no put-back."""
    out = _notification_queue[:]
    _notification_queue.clear()
    return out

def format_task_notification(
    agent_id: str, agent_type: str, status: str,
    output_file: str, final_text: Optional[str] = None,
    error: Optional[str] = None, duration_ms: Optional[int] = None,
    total_tokens: Optional[int] = None, tool_use_count: Optional[int] = None,
    worktree_path: Optional[str] = None, worktree_branch: Optional[str] = None,
    description: Optional[str] = None,
) -> str:
    lines = ["<task-notification>"]
    lines.append(f"  <task_id>{agent_id}</task_id>")
    lines.append(f"  <agent_type>{agent_type}</agent_type>")
    lines.append(f"  <status>{status}</status>")
    if description:
        lines.append(f"  <description>{description}</description>")
    lines.append(f"  <output_file>{output_file}</output_file>")
    if final_text:
        lines.extend(["  <result>", final_text, "  </result>"])
    if error:
        lines.append(f"  <error>{error}</error>")
    usage_bits = []
    if total_tokens is not None: usage_bits.append(f"tokens={total_tokens}")
    if tool_use_count is not None: usage_bits.append(f"tools={tool_use_count}")
    if duration_ms is not None: usage_bits.append(f"duration_ms={duration_ms}")
    if usage_bits:
        lines.append(f"  <usage>{' '.join(usage_bits)}</usage>")
    if worktree_path:
        lines.append(f"  <worktree_path>{worktree_path}</worktree_path>")
        if worktree_branch:
            lines.append(f"  <worktree_branch>{worktree_branch}</worktree_branch>")
    lines.append("</task-notification>")
    return "\n".join(lines)

# Demo
notif_text = format_task_notification(
    agent_id="agent-abc123",
    agent_type="Explore",
    status="completed",
    output_file="/tmp/agent-abc123.jsonl",
    final_text="Found 3 usages of resolveAgentTools in src/agents/",
    duration_ms=3200,
    total_tokens=4500,
    tool_use_count=2,
    description="search task",
)
enqueue_pending_notification("task-notification", notif_text)
print(f"Queue depth: {len(_notification_queue)}")
drained = drain_pending_notifications()
print(f"Drained {len(drained)} notification(s)")
print("\n--- Notification text the LLM will see ---")
print(drained[0].text)
print(f"\nQueue after drain: {len(_notification_queue)}")

## 18. Background Agent Lifecycle (Async)

`src/agents/runAsyncAgent.ts`

`runAsyncAgentLifecycle()` is the fire-and-forget wrapper that ties all subsystems together for background agents.

**Key design decisions:**
- **Own AbortController** — ESC on parent does not kill background agents
- **`shouldAvoidPermissionPrompts: true`** — "ask" decisions auto-deny to avoid blocking parent UI
- **`sessionIdOverride`** — pins the sub-session to the public `agentId` for transcript addressability
- **Never throws** — called with `void runAsyncAgentLifecycle(...)`, errors are caught and recorded
- **Dual writes** — progress events go to both the JSONL output file AND the in-memory `asyncAgentStore`

```
register  →  runChildAgent(shouldAvoidPermissionPrompts=True)
    │              │
    │         onProgress events
    │              ├── appendTaskOutput(outputFile)     ← JSONL for parent's Read tool
    │              └── updateAsyncAgentProgress(store)  ← live UI updates
    │
    └── On done:
           cleanupWorktreeIfNeeded()
           completeAsyncAgent(store)  OR  failAsyncAgent(store)
           enqueuePendingNotification()
```

In [ ]:
def run_async_agent_lifecycle_stub(
    agent_id: str,
    agent_type: str,
    prompt: str,
    output_file: str,
    description: Optional[str] = None,
):
    """
    Simplified demonstration of the lifecycle flow.
    Real version: async, calls runChildAgent (LLM loop), writes to disk JSONL.
    Simplified here: simulates the state transitions synchronously.
    """
    entry = register_async_agent(
        agent_id=agent_id,
        agent_type=agent_type,
        prompt=prompt,
        output_file=output_file,
        description=description,
    )
    output_log = []

    # Header record
    output_log.append({"type": "started", "agentType": entry.agentType, "prompt": prompt})

    # Simulate progress events the real onProgress callback would receive
    progress_events = [
        {"type": "tool_use_start", "toolName": "Glob"},
        {"type": "tool_use_done",  "toolName": "Glob", "isError": False},
        {"type": "tool_use_start", "toolName": "Grep"},
        {"type": "tool_use_done",  "toolName": "Grep", "isError": False},
        {"type": "text", "text": "Found usages in src/agents/resolveAgentTools.ts"},
        {"type": "turn_usage", "cumulativeUsage": {"input_tokens": 1200, "output_tokens": 300,
                                                    "cache_read_input_tokens": 0, "cache_creation_input_tokens": 0},
         "turnCount": 1},
    ]

    for ev in progress_events:
        output_log.append(ev)
        if ev["type"] == "tool_use_start":
            update_async_agent_progress(agent_id, lastToolName=ev["toolName"])
        elif ev["type"] == "tool_use_done":
            cur = _async_agent_entries[agent_id]
            update_async_agent_progress(agent_id, toolUseCount=cur.toolUseCount + 1)
        elif ev["type"] == "turn_usage":
            u = ev["cumulativeUsage"]
            total = u["input_tokens"] + u["output_tokens"]
            update_async_agent_progress(agent_id, totalTokens=total,
                                         inputTokens=u["input_tokens"], outputTokens=u["output_tokens"],
                                         turnCount=ev["turnCount"])

    # Simulate completion
    sim_result = AgentRunResult(
        agentType=agent_type,
        finalText="Found 3 usages in src/agents/resolveAgentTools.ts at lines 44, 62, 77.",
        messages=[], totalToolUseCount=2, totalDurationMs=2100,
        totalTokens=1500, inputTokens=1200, outputTokens=300, turnCount=1, reason="completed",
    )

    # No worktree in this demo, so cleanup is a no-op
    output_log.append({"type": "completed", "reason": sim_result.reason,
                        "finalText": sim_result.finalText, "durationMs": sim_result.totalDurationMs})

    complete_async_agent(agent_id, sim_result)

    # Enqueue notification
    notif = format_task_notification(
        agent_id=agent_id, agent_type=agent_type, status="completed",
        output_file=output_file, final_text=sim_result.finalText,
        duration_ms=sim_result.totalDurationMs, total_tokens=sim_result.totalTokens,
        tool_use_count=sim_result.totalToolUseCount, description=description,
    )
    enqueue_pending_notification("task-notification", notif)

    return output_log, sim_result

bg_id = "bg-search-001"
output_log, sim_result = run_async_agent_lifecycle_stub(
    agent_id=bg_id, agent_type="Explore",
    prompt="Find all usages of resolveAgentTools in the codebase",
    output_file=f"/tmp/{bg_id}.jsonl",
    description="find usages",
)

print("Output log (mirrors JSONL written to disk):")
for entry in output_log:
    print(f"  {entry}")

print(f"\nStore entry status : {_async_agent_entries[bg_id].status}")
print(f"Pending notifications: {len(_notification_queue)}")
print("\n--- Draining notification queue (at next user submission) ---")
for n in drain_pending_notifications():
    print(n.text)

## Summary

| Notebook cell | Source file(s) | Core concept |
|---|---|---|
| 2. Data Models | `src/agents/types.ts` | `AgentDefinition` + `AgentRunResult` |
| 3. Agent Registry | `src/agents/registry.ts` | Write-once, read-many dict; project > user > built-in |
| 4. Built-in Agents | `src/agents/builtIn/explore.ts`, `generalPurpose.ts` | Explore (disallowed Write/Edit), general-purpose (wildcard) |
| 5. Agent Loading | `src/agents/loadAgentsDir.ts` | YAML frontmatter parsing; two scopes in parallel |
| 6. Bootstrap | `src/agents/bootstrap.ts` | `setAgents([...builtins, ...custom])` |
| 7. Tool Resolution | `src/agents/resolveAgentTools.ts` | Strip Agent + disallowed → wildcard or intersect |
| 8. Prompt Injection | `src/agents/promptInjection.ts` | `<system-reminder>` listing + creation template |
| 9. System Prompt | `src/context/systemPrompt.ts` | Static zone + dynamic zone; parallel fetching |
| 10. AGENT.md | `src/context/claudeMd.ts` | Root-to-cwd hierarchy; HTML-comment stripping |
| 11. Memory | `src/context/memory/memdir.ts` | SHA-256 project key; MEMORY.md index; fuzzy dedup |
| 12. Compaction | `src/context/compaction.ts` | Micro-compact (cheap) then full summarize (API call) |
| 13. Auto-Compact | `src/context/autoCompact.ts` | Circuit breaker; token warning states; recursion guard |
| 14. Plan Mode | `src/context/plans.ts`, `planAttachments.ts` | Random slug plan files; throttled user-message injection |
| 15. Session Persistence | `src/session/storage.ts` | JSONL transcript; compaction marker for fast restore |
| 16. Async Agent Store | `src/state/asyncAgentStore.ts` | Pub/sub state; independent AbortController |
| 17. Notification Store | `src/state/notificationStore.ts` | FIFO queue; drain at next user submission |
| 18. Background Lifecycle | `src/agents/runAsyncAgent.ts` | Fire-and-forget; dual writes; never throws |

**Key architectural invariants:**
- Sub-agents **cannot spawn sub-sub-agents** (Agent tool physically removed from pool)
- Background agents use their **own AbortController** (ESC on parent doesn't kill them)
- Compaction **never touches the last 8 messages** (preserves recent tool call pairs)
- Auto-compaction has a **circuit breaker** (3 consecutive failures → stop trying)
- Session restore starts from the **last compaction marker** (pre-compaction context is already in the summary)